<div style="background-color:#e6f2ff; padding:20px; border-radius:10px;">
<img style="float:left; margin-right:20px;" src='Figures/alinco.png' width="120"/>
<h1 style="color:#000047;">Actividad 1: Implementación de un modelo por Random Forest</h1>
<br style="clear:both"/>
</div>

<div style="border-left:4px solid #000047; padding:10px; margin-top:10px; background:#f5f5f5;">
<b>Objetivo:</b> En esta actividad construiras un clasificadore basados en **Random Forest** para predecir la **aceptabilidad/seguridad de un automóvil**. 
</div>

<div style="margin-top:10px;">
<b>Instrucciones generales:</b>
<ul>
<li>Entrenar dos modelos (con pocos y con muchos árboles) para observar cómo mejora la exactitud, demostrar el proceso de **selección de características** basado en la importancia de variables, y reconstruir el modelo con las variables más relevantes. </li>

</ul>
</div>

**Random Forest** (bosque aleatorio) es un algoritmo de **aprendizaje supervisado** basado en **ensamble** (*ensemble learning*). Tiene dos variantes: una para **clasificación** y otra para **regresión**. Es uno de los algoritmos más flexibles y fáciles de usar.

Construye **múltiples árboles de decisión** sobre distintas muestras de los datos, obtiene la predicción de cada árbol y selecciona la mejor solución mediante **votación**. Además, es un excelente indicador de la **importancia de las variables**.

> Al combinar muchos árboles se forma un "bosque"; de ahí el nombre **Random Forest**. Cuantos **más árboles**, mayor suele ser la exactitud (hasta estabilizarse).

#### Selección de características con Random Forest

El Random Forest permite **ordenar la importancia** de las variables. Durante el entrenamiento, se registra el **error out-of-bag (OOB)** y se promedia sobre el bosque.

Para medir la importancia de la característica *j*, se **permutan** sus valores en los datos y se recalcula el error OOB sobre ese conjunto perturbado. La **importancia** es la diferencia promedio de error OOB antes y después de la permutación, normalizada por su desviación estándar.

Las características que producen valores **grandes** de esta puntuación son más importantes. Con base en ella, conservaremos las más relevantes y descartaremos las menos útiles.

**Planteamiento del problema**

El objetivo es **predecir la aceptabilidad/seguridad de un automóvil** con un clasificador Random Forest en Python y scikit-learn, usando el [**Car Evaluation Data Set**](http://archive.ics.uci.edu/ml/datasets/Car+Evaluation) de la UCI.

El **Car Evaluation Data Set** relaciona la evaluación del automóvil (`class`) con **seis atributos de entrada**:

| Variable | Descripción | Valores |
|---|---|---|
| `buying` | Precio de compra | vhigh, high, med, low |
| `maint` | Costo de mantenimiento | vhigh, high, med, low |
| `doors` | Número de puertas | 2, 3, 4, 5more |
| `persons` | Capacidad de personas | 2, 4, more |
| `lug_boot` | Tamaño de la cajuela | small, med, big |
| `safety` | Seguridad estimada | low, med, high |
| `class` | **Objetivo:** evaluación | unacc, acc, good, vgood |

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Dataset real de la UCI (se descarga al vuelo)
#url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'
#df = pd.read_csv(url, header=None)
# Alternativa local: df = pd.read_csv('C:/datasets/car.data', header=None)

df = pd.read_csv('car.data', header=None)

## 1.- Análisis exploratorio de datos

In [4]:
# Primeras filas (columnas aún sin nombre)
df.head()

,0,1,2,3,4,5,6
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [5]:
col_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df.columns = col_names
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   buying    1728 non-null   object
 1   maint     1728 non-null   object
 2   doors     1728 non-null   object
 3   persons   1728 non-null   object
 4   lug_boot  1728 non-null   object
 5   safety    1728 non-null   object
 6   class     1728 non-null   object
dtypes: object(7)
memory usage: 94.6+ KB


In [7]:
# Distribución de frecuencias
for col in col_names:
    print(df[col].value_counts())
    print('=' * 40)

buying
vhigh    432
high     432
med      432
low      432
Name: count, dtype: int64
maint
vhigh    432
high     432
med      432
low      432
Name: count, dtype: int64
doors
2        432
3        432
4        432
5more    432
Name: count, dtype: int64
persons
2       576
4       576
more    576
Name: count, dtype: int64
lug_boot
small    576
med      576
big      576
Name: count, dtype: int64
safety
low     576
med     576
high    576
Name: count, dtype: int64
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


### Variable objetivo `class` y valores faltantes
La variable objetivo es **ordinal** (unacc < acc < good < vgood).

In [8]:
df['class'].value_counts()

class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64

In [9]:
faltante = df.isnull().sum()
print(f"Total de valores nulos: ,{faltante}")

Total de valores nulos: ,buying      0
maint       0
doors       0
persons     0
lug_boot    0
safety      0
class       0
dtype: int64


## 2. Definir variables predictoras y objetivo

In [10]:
#Variable Objetivo
y = df['class']

#Variables Predictoras [todas excepto 'class']
X = df.drop(['class'], axis=1)

## 3. Dividir en entrenamiento y prueba

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Entrenamiento:', X_train.shape, '| Test:', X_test.shape)

Entrenamiento: (1382, 6) | Test: (346, 6)


## 4. Ingeniería de características (codificación)

Todas las variables son **categóricas ordinales**; las convertimos a números con `OrdinalEncoder` de scikit-learn (sin dependencias externas).

In [12]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
X_train = pd.DataFrame(encoder.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(encoder.transform(X_test), columns=X.columns, index=X_test.index)

X_train.head()

,buying,maint,doors,persons,lug_boot,safety
48,3.0,3.0,1.0,2.0,1.0,1.0
468,0.0,3.0,1.0,1.0,2.0,1.0
155,3.0,0.0,1.0,2.0,2.0,0.0
1721,1.0,1.0,3.0,2.0,2.0,0.0
1208,2.0,1.0,0.0,2.0,2.0,0.0


In [13]:
X_test.head()

,buying,maint,doors,persons,lug_boot,safety
599,0.0,0.0,2.0,0.0,1.0,0.0
1201,2.0,1.0,0.0,1.0,1.0,2.0
628,0.0,0.0,3.0,0.0,0.0,2.0
1498,1.0,0.0,3.0,1.0,1.0,2.0
1263,2.0,1.0,2.0,2.0,1.0,1.0


## 5. Random Forest con pocos árboles (modelo 1)

Entrena un bosque con **10 árboles** para tener una referencia.


In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

##ÁRbol de referencia
#tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

# Modelo 1: Random Forest con 10 Árboles con OOB score activado
rf_mod1 = RandomForestClassifier(
    n_estimators=10, max_features='sqrt', oob_score=True,
    random_state=42, n_jobs=-1,
).fit(X_train, y_train)

print('Random Forest con 10 Árboles -> test accuracy:', round(accuracy_score(y_test, rf_mod1.predict(X_test))*100, 1))
print('Random Forest con 10 Árboles -> OOB score   :', round((rf_mod1.oob_score_)*100, 1))

ValueError: could not convert string to float: 'vhigh'

In [12]:
y_pred = rf_mod1.predict(X_train)

NameError: name 'rf_mod1' is not defined

## 6. Random Forest con muchos arboles `n_estimators=100`

Entrena un modelo con **100 árboles** y comenta lo que puedas observar.

In [15]:

# Modelo 2: Random Forest con 100 Árboles con OOB score activado
rf_mod2 = RandomForestClassifier(
    n_estimators=100, max_features='sqrt', oob_score=True,
    random_state=10, n_jobs=-1,
).fit(X_train, y_train)

print('Random Forest con 100 Árboles -> test accuracy:', round(accuracy_score(y_test, rf_mod2.predict(X_test))*100, 1))
print('Random Forest con 100 Árboles -> OOB score   :', round((rf_mod2.oob_score_)*100, 1))

Random Forest con 100 Árboles -> test accuracy: 96.1
Random Forest con 100 Árboles -> OOB score   : 97.4


#### Comentarios
Se mantienen todos los parametros a excepción de n_estimator (número de árboles), en el Modelo 1 son 10 árboles y en el Modelo 2 son 100 árboles.

* Modelo 1--> 10  árboles: 95.3% Test Accuracy (exactitud)
* Modelo 2--> 100 árboles: 96.1% Test Accuracy (exactitud)
Diferencia a favor del modelo 2 con +0.88%, indicando un ligero mejoramiento en la exactitud.

* Modelo 1--> 10  árboles: 92.1% OOB score
* Modelo 2--> 100 árboles: 97.4% OOB score
Diferencia a favor del modelo 2 con +5.3%, indicando reducción en las variaciones de cada árbol quedando a 2.6% de un 100%.

Con ello, el modelo2 es más preciso y estable, ligeramente muestra un mejor desempeño. 

Podría considerarse quedarse con el Modelo 1, pues el incremento de la exactitud es mínima considerando que se incrementa de 10 a 100 árboles. 


## 7. Importancia de las características

Usa el atributo `feature_importances_` del bosque para ordenar las variables por relevancia.

In [16]:
# Importancia de variables del bosque con 10 Árboles, Modelo 1
imp_var1 = (pd.Series(rf_mod1.feature_importances_, index=X.columns).sort_values(ascending=False)*100).round(1)
print('\nMODELO 1 --> Importancia de Variables Ordenadas, según su relevancia:\n', imp_var1)
print('\nSuma de Importancias:', round(imp_var1.sum(),1),'%')


MODELO 1 --> Importancia de Variables Ordenadas, según su relevancia:
 safety      25.7
persons     22.5
buying      16.9
maint       16.8
lug_boot    10.7
doors        7.5
dtype: float64

Suma de Importancias: 100.1 %


In [17]:
# Importancia de variables del bosque con 100 Árboles, Modelo 2
imp_var2 = (pd.Series(rf_mod2.feature_importances_, index=X.columns).sort_values(ascending=False)*100).round(1)
print('\nMODELO 2 --> Importancia de Variables Ordenadas, según su relevancia:\n', imp_var2)
print('\nSuma de Importancias:', round(imp_var2.sum(),1),'%')


MODELO 2 --> Importancia de Variables Ordenadas, según su relevancia:
 safety      26.9
persons     21.9
buying      20.0
maint       15.6
lug_boot     8.8
doors        6.8
dtype: float64

Suma de Importancias: 100.0 %


## 8. Visualizar la importancia de las características

In [ ]:
plt.figure(figsize=(8, 4))
imp_var1.plot.barh(color='#2E9AFE')
plt.xlabel('Importancia (reducción de impureza)')
plt.title('Importancia de variables — Random Forest (Automóviles)')
plt.tight_layout()
plt.show()

In [18]:
import matplotlib.pyplot as plt
print(pd.__version__)
print("Matplotlib OK")

2.3.3
Matplotlib OK


## 9. Reconstruir el modelo con las características seleccionadas

Elimina la variable **menos importante**, reconstruye el modelo y compara la exactitud.

In [ ]:
#La variable si 

In [19]:
X_var_elim = X.drop(['doors'], axis=1)

In [20]:
# Nuevo conjunto de variables sin 'doors'
print(X_var_elim.columns)

Index(['buying', 'maint', 'persons', 'lug_boot', 'safety'], dtype='object')


In [24]:
#Vuelvo a dividir las variables para entrenamiento y prueba, 
X_train, X_test, y_train, y_test = train_test_split(X_var_elim, y, test_size=0.33, random_state=42)
print('Entrenamiento:', X_train.shape, '| Test:', X_test.shape)

X_train.head()

Entrenamiento: (1157, 5) | Test: (571, 5)


,buying,maint,persons,lug_boot,safety
48,vhigh,vhigh,more,med,low
468,high,vhigh,4,small,low
155,vhigh,high,more,small,high
1721,low,low,more,small,high
1208,med,low,more,small,high


In [22]:
#Utilizamos el OrdinalEncord
encoder = OrdinalEncoder()
X_train = pd.DataFrame(encoder.fit_transform(X_train), columns=X_var_elim.columns, index=X_train.index)
X_test = pd.DataFrame(encoder.transform(X_test), columns=X_var_elim.columns, index=X_test.index)

X_train.head()

,buying,maint,persons,lug_boot,safety
48,3.0,3.0,2.0,1.0,1.0
468,0.0,3.0,1.0,2.0,1.0
155,3.0,0.0,2.0,2.0,0.0
1721,1.0,1.0,2.0,2.0,0.0
1208,2.0,1.0,2.0,2.0,0.0


In [23]:
# Utilizamos el Modelo 2
# Modelo 3: Random Forest con 100 Árboles con OOB score activado (sin la variable 'doors')
rf_mod3 = RandomForestClassifier(
    n_estimators=100, max_features='sqrt', oob_score=True,
    random_state=10, n_jobs=-1,
).fit(X_train, y_train)

print('Random Forest con 100 Árboles -> test accuracy:', round(accuracy_score(y_test, rf_mod3.predict(X_test))*100, 1))
print('Random Forest con 100 Árboles -> OOB score   :', round((rf_mod3.oob_score_)*100, 1))

Random Forest con 100 Árboles -> test accuracy: 92.6
Random Forest con 100 Árboles -> OOB score   : 94.1


### Comentarios
Los resultados del Modelo 3 con la eliminación de la variable 'doors' que es la menos importante:

* Modelo 2 tiene seis variables, Modelo 3 tiene cinco variables.
##### Test Accuracy (exactitud)
* Modelo 2--> 100 árboles y 6 variables: 96.1%.
* Modelo 3--> 100 árboles y 5 variables: 92.6%.
Diferencia a favor del modelo 2 con +3.5%, indicando un decremento en la exactitud en el Modelo 3.

##### OOB score
* Modelo 2--> 100 árboles:97.4% OOB score
* Modelo 3--> 100 árboles: 94.1% OOB score
Diferencia a favor del modelo 2 con +3.3%, indicando mayor aumento en las variaciones de cada árbol en el Modelo 3.

Aunque se eliminó la variable menos importante 'doors' del modelo, tuvo un gran impacto en los resultados del Test y OOB score, por lo que se puede decir que si aporta información útil al modelo. Por lo que disminutó el desempeño del Modelo.

## 10. Crea un Reporte de clasificación

Muestra **precision**, **recall**, **f1-score** y **support** por clase.